# **PyMC Price Target Model**

## Price Target Achievement — Hierarchical Student-t Regression

Schema-aligned with:
- MV: `pml.mv_pymc_price_target`
- Catalogue: `SELECT * FROM pml.vw_pymc_feature_catalogue WHERE model_target = 'price_target'`

Likelihood:
- `observed_target_pct ~ StudentT(ν, μ_isin, σ_isin)` (heavy-tailed analyst noise)
- `μ_isin = α_sector[sector] + X_std · β`
- `σ_isin = σ_base / sqrt(n_analysts)` (precision-weighted by analyst count)


In [ ]:
%%sql
SELECT * FROM pml.mv_pymc_price_target

In [ ]:
%%sql
SELECT *
FROM pml.vw_pymc_feature_catalogue
WHERE model_target = 'price_target'
ORDER BY pymc_role, feature_role, feature_alias

## 1. Notebook Setup & Imports

Shared imports for EDA, schema-aligned summary statistics, PyMC modelling, and prior/posterior predictive diagnostics. The `price_target_df` and `feature_catalogue` DataFrames produced by the two `%%sql` cells above are reused throughout.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import arviz as az
except ImportError:  # ArviZ 1.0 split packages
    import arviz_base as az  # type: ignore

import pymc as pm
import pytensor.tensor as pt

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# --- Dark theme for all plots ------------------------------------------------
# Matplotlib dark background + seaborn dark grid keep ArviZ / pandas / seaborn
# figures visually consistent. Try ArviZ's bundled dark stylesheet first; if
# unavailable, fall back to matplotlib's built-in 'dark_background'.
plt.style.use('dark_background')
try:
    az.style.use('arviz-darkgrid')
except (OSError, ValueError, AttributeError):
    pass
sns.set_theme(style='darkgrid', context='notebook',
              rc={
                  'figure.facecolor': '#1e1e1e',
                  'axes.facecolor': '#2a2a2a',
                  'savefig.facecolor': '#1e1e1e',
                  'axes.edgecolor': '#cccccc',
                  'axes.labelcolor': '#e6e6e6',
                  'xtick.color': '#e6e6e6',
                  'ytick.color': '#e6e6e6',
                  'text.color': '#e6e6e6',
                  'grid.color': '#555555',
              })
plt.rcParams['figure.dpi'] = 110

print(f'price_target_df : {price_target_df.shape}')
print(f'feature_catalogue: {feature_catalogue.shape}  (model_target=price_target)')
assert (feature_catalogue['model_target'] == 'price_target').all(), \
    "feature_catalogue must be pre-filtered to model_target='price_target'"


## 2. Exploratory Data Analysis (EDA) — `price_target_df`

Schema-aligned EDA driven by `feature_catalogue` (the `vw_pymc_feature_catalogue` view for `model_target = 'price_target'`). We split columns by `pymc_role` (`response`, `mutable_predictor`, `coord`) and `feature_role` to drive shape, missingness, and distribution checks.


In [ ]:
# Map feature_catalogue -> columns actually present in price_target_df
catalogue = feature_catalogue.copy()
catalogue['present'] = catalogue['feature_alias'].isin(price_target_df.columns)

role_summary = (
    catalogue.groupby(['pymc_role', 'feature_role'])['present']
    .agg(n_columns='size', n_present='sum')
    .reset_index()
)
role_summary


In [ ]:
# Resolve column groups by pymc_role
present = catalogue.loc[catalogue['present']]
PREDICTOR_COLS = present.loc[present['pymc_role'] == 'mutable_predictor', 'feature_alias'].tolist()
COORD_COLS = present.loc[present['pymc_role'] == 'coord', 'feature_alias'].tolist()
RESPONSE_COLS = present.loc[present['pymc_role'].isin(['response', 'observed']), 'feature_alias'].tolist()

# Canonical fallbacks for the price-target MV
for col in ('observed_target_pct', 'observed_target_pct_med', 'n_analysts', 'last_price'):
    if col in price_target_df.columns and col not in RESPONSE_COLS + COORD_COLS:
        if col.startswith('observed_'):
            RESPONSE_COLS.append(col)

# Classification coords (all 9 from the new MV)
CLASSIFICATION_COORDS = [c for c in (
    'isin', 'ticker', 'region', 'country', 'trading_country',
    'exchange', 'unit', 'style_class', 'size_class', 'sector', 'industry'
) if c in price_target_df.columns]

# Build source-column -> MV alias mapping from the catalogue. The catalogue
# view now exposes the per-model `feature_alias` populated by
# pml.pml_df_feature_alias, so downstream code can address columns by their
# materialized-view alias (e.g. feat_eps_fy1e, observed_target_pct, n_beats,
# region/country/sector classification coords).
if 'feature_alias' in catalogue.columns:
    ALIAS_MAP = (
        present.dropna(subset=['feature_alias'])
        .set_index('column_name')['feature_alias']
        .to_dict()
    )
else:
    ALIAS_MAP = {c: c for c in present['column_name']}

print(f'#predictors : {len(PREDICTOR_COLS)}')
print(f'#coords     : {len(COORD_COLS)}  -> {COORD_COLS}')
print(f'#response   : {len(RESPONSE_COLS)} -> {RESPONSE_COLS}')
print(f'#classification : {len(CLASSIFICATION_COORDS)} -> {CLASSIFICATION_COORDS}')
print(f'#aliases    : {len(ALIAS_MAP)} (source_column -> mv_alias)')


In [ ]:
# 2.1 Shape, dtypes, missingness overview
missing = price_target_df.isna().mean().sort_values(ascending=False)
eda_overview = pd.DataFrame({
    'dtype': price_target_df.dtypes.astype(str),
    'missing_pct': missing.round(4),
    'n_unique': price_target_df.nunique(dropna=True),
})
eda_overview.head(20)


In [ ]:
# 2.2 Response distribution: observed_target_pct + n_analysts
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

price_target_df['observed_target_pct'].dropna().plot.hist(
    bins=60, ax=axes[0], edgecolor='#1e1e1e', color='#4FC3F7'
)
axes[0].axvline(0, color='#FF5252', ls='--', lw=1)
axes[0].set_title('observed_target_pct  (analyst implied upside, %)')
axes[0].set_xlabel('target_pct_avg')

n_analysts_s = price_target_df['n_analysts'].dropna().astype(int)
n_analysts_s.plot.hist(bins=range(0, int(n_analysts_s.max()) + 2),
                       ax=axes[1], edgecolor='#1e1e1e', color='#66BB6A')
axes[1].set_title('n_analysts (price_target_num)')
plt.tight_layout()
plt.show()


In [ ]:
# 2.3 Classification coord cardinality (region / country / sector / industry / ...)
for col in CLASSIFICATION_COORDS:
    if col in ('isin', 'ticker'):
        continue
    counts = price_target_df[col].value_counts(dropna=False).head(15)
    fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(counts))))
    counts.sort_values().plot.barh(ax=ax)
    ax.set_title(f'Top categories — {col}')
    plt.tight_layout()
    plt.show()


In [ ]:
# 2.4 Predictor correlation heatmap (numeric predictors only)
num_predictors = [c for c in PREDICTOR_COLS
                  if pd.api.types.is_numeric_dtype(price_target_df[c])]

if len(num_predictors) < 2:
    print(f'⚠️ Predictor correlation heatmap skipped — '
          f'only {len(num_predictors)} numeric predictor(s) available '
          f'(need ≥ 2). Check that `feature_catalogue` tags '
          f'`mutable_predictor` columns that actually exist in `price_target_df`.')
else:
    corr = price_target_df[num_predictors].corr(numeric_only=True)
    side = min(14, 0.4 * len(num_predictors) + 4)
    fig, ax = plt.subplots(figsize=(side, side))
    # Only render tick labels when the matrix is small enough to read them.
    show_labels = len(num_predictors) <= 30
    sns.heatmap(
        corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        xticklabels=show_labels, yticklabels=show_labels,
        square=True, ax=ax,
    )
    if show_labels:
        # Pin tick positions to a FixedLocator before relabelling so Matplotlib
        # does not warn about a mismatch between auto-located ticks and the
        # explicit labels we supply (robust to any variable number of features).
        x_locs = ax.get_xticks()
        x_labels = [t.get_text() for t in ax.get_xticklabels()]
        ax.set_xticks(x_locs)
        ax.set_xticklabels(x_labels, rotation=75, ha='right')

        y_locs = ax.get_yticks()
        y_labels = [t.get_text() for t in ax.get_yticklabels()]
        ax.set_yticks(y_locs)
        ax.set_yticklabels(y_labels, rotation=0, va='center')
    ax.set_title(f'Predictor correlation ({len(num_predictors)} feat_* features)')
    plt.tight_layout()
    plt.show()


## 3. Summary Statistics by Feature Role

Statistics are grouped using `feature_catalogue.feature_role` (e.g. `predictor`, `target`, `categorical`) and `category` (e.g. `eps_revisions`, `fiscal_calendar`, `classification`). This mirrors the catalogue-driven approach used in `pymc_expected_returns_model.ipynb` §13 (catalogue-driven coverage check).


In [ ]:
def summarize_by_role(df: pd.DataFrame, cat: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-column statistics, joined with feature_catalogue metadata.

    The catalogue uses two identifiers:
      - ``column_name``   — source column in ``pml.pml_df``
      - ``feature_alias`` — MV/view alias actually exposed in ``price_target_df``
        (populated from ``pml.pml_df_feature_alias`` per ``model_target``).

    Rows in ``price_target_df`` are addressable by alias, so we resolve presence by
    alias first and fall back to ``column_name`` for catalogue rows that pass
    the source column through unchanged (most ``coord`` entries).
    """
    numeric_fields = ['mean', 'std', 'p05', 'median', 'p95', 'skew', 'kurt']

    # 1. Resolve, for each catalogue row, which df column (if any) backs it.
    has_alias = 'feature_alias' in cat.columns
    cat = cat.copy()
    cat['resolved_column'] = (
        cat['feature_alias'].where(has_alias and cat['feature_alias'].notna(),
                                   cat['column_name'])
        if has_alias else cat['column_name']
    )
    # Keep only rows whose resolved name exists in the dataframe, de-duplicated
    # (a single MV alias can be referenced by multiple source columns).
    resolved = (
        cat.loc[cat['resolved_column'].isin(df.columns)]
        .drop_duplicates(subset=['resolved_column'])
    )

    # 2. Per-column statistics keyed by the resolved (df-side) column name.
    rows = []
    for col in resolved['resolved_column']:
        s = df[col]
        rec = {
            'resolved_column': col,
            'dtype': str(s.dtype),
            'n': int(s.notna().sum()),
            'missing_pct': float(s.isna().mean()),
            'n_unique': int(s.nunique(dropna=True)),
        }
        # Ensure numeric stat columns always exist (NaN for non-numeric dtypes)
        for f in numeric_fields:
            rec[f] = np.nan
        if pd.api.types.is_numeric_dtype(s):
            desc = s.describe(percentiles=[0.05, 0.5, 0.95])
            rec.update({
                'mean': float(desc['mean']),
                'std': float(desc['std']),
                'p05': float(desc['5%']),
                'median': float(desc['50%']),
                'p95': float(desc['95%']),
                'skew': float(s.skew(skipna=True)),
                'kurt': float(s.kurt(skipna=True)),
            })
        rows.append(rec)

    stats = pd.DataFrame(rows)

    # 3. Merge metadata back. Use ``resolved_column`` as the join key so every
    #    feature category (coord, mutable_predictor, constant_data, observed)
    #    keeps its catalogue annotations. Rename the catalogue's source
    #    ``column_name`` to ``source_column_name`` BEFORE the merge so it does
    #    not collide with the alias-based ``column_name`` we surface below.
    meta_cols = [c for c in
                 ['column_name', 'pymc_role', 'feature_role',
                  'feature_alias', 'category', 'data_type']
                 if c in cat.columns]
    meta_df = resolved[['resolved_column', *meta_cols]].rename(
        columns={'column_name': 'source_column_name'}
    )
    out = stats.merge(meta_df, on='resolved_column', how='left')

    # 4. Surface a stable ``column_name`` (= MV/df-side alias) for downstream
    #    code that filters price_target_df by alias (e.g. the target box plot).
    out = out.rename(columns={'resolved_column': 'column_name'})

    # Reorder for readability
    leading = ['column_name', 'feature_alias', 'source_column_name',
               'pymc_role', 'feature_role',
               'category', 'data_type', 'dtype', 'n', 'missing_pct', 'n_unique']
    leading = [c for c in leading if c in out.columns]
    return out[leading + [c for c in out.columns if c not in leading]]


summary_stats = summarize_by_role(price_target_df, catalogue)
summary_stats.head(50)


In [ ]:
# 3.1 Roll-up by (pymc_role, feature_role, category)
rollup = (
    summary_stats
    .groupby(['pymc_role', 'category', 'feature_role'], dropna=False)
    .agg(
        n_columns=('feature_alias', 'size'),
        avg_missing_pct=('missing_pct', 'mean'),
        mean_of_means=('mean', 'mean'),
        mean_of_stds=('std', 'mean'),
    )
    .round(4)
    .reset_index()
    .sort_values(['pymc_role', 'category', 'feature_role'])
)
rollup


In [ ]:
# 3.2 Visualise distribution of target-family predictors per category
#     Iterate feature_aliases for the 'target' feature_role and map them
#     onto price_target_df columns (the MV exposes columns by feature_alias).
rev_aliases = (
    summary_stats.loc[summary_stats['feature_role'] == 'target',
    'feature_alias']
    .dropna()
    .unique()
    .tolist()
)
rev_cols = [a for a in rev_aliases if a in price_target_df.columns
            and pd.api.types.is_numeric_dtype(price_target_df[a])]
if rev_cols:
    melted = price_target_df[rev_cols].melt(var_name='feature_role', value_name='value').dropna()
    fig, ax = plt.subplots(figsize=(min(14, 0.4 * len(rev_cols) + 4), 5))
    sns.boxplot(data=melted, x='feature_role', y='value', ax=ax, showfliers=True)

    # --- Option A: pin tick positions, then set labels (fully explicit) ---
    # Works for any variable number of categories. `ax.get_xticks()` returns
    # the locations seaborn used; passing them through FixedLocator-equivalent
    # `set_xticks` silences the warning.
    tick_locs = ax.get_xticks()
    tick_labels = [t.get_text() for t in ax.get_xticklabels()]
    ax.set_xticks(tick_locs)
    ax.set_xticklabels(tick_labels, rotation=75, ha='right')

    # --- Option B (equivalent, even shorter): rotate in place ---
    # plt.setp(ax.get_xticklabels(), rotation=75, ha='right')

    ax.set_title('Target-family predictor distributions')
    plt.tight_layout()
    plt.show()
else:
    print('No target-role predictors present in price_target_df.')


## 4. Build PyMC-Aligned Data Containers

Hierarchical Student-t regression on `observed_target_pct`, schema-aligned with `PriceTargetAchievement` (see `probabilistic_ml_model.pymc_models.PriceTargetModel`). The likelihood is a heavy-tailed Student-t on the analyst-implied upside (`observed_target_pct = target_pct_avg`), with the per-isin scale shrunk by `1 / sqrt(n_analysts)` so that broadly-covered names carry more weight. `pm.Data` container names mirror `pml.mv_pymc_price_target` columns (and therefore `vw_pymc_feature_catalogue.feature_alias`) so that `pm.set_data` round-trips against the live MV.

**Schema alignment with `pml.mv_pymc_price_target`.** The model coords now include every categorical column emitted by the materialized view (DDL columns `region`, `country`, `trading_country`, `exchange`, `unit`, `style_class`, `size_class`, `sector`, `industry`). Each coord is registered as a `pm.Model` dimension and as a `pm.Data('<coord>_idx', …, dims='isin')` integer-index container, so posterior predictive checks can be conditioned on any of them via `pm.set_data`. Hierarchical partial-pooling intercepts are attached to `sector`, `size_class`, and `style_class` (the highest-signal, lowest-cardinality coords); the remaining coords are available for downstream slicing without inflating the parameter count.


In [ ]:
# 4.0  Filter rows with a usable response
DATE_COLS = [
    'income_statement_report_date',
    'next_earnings',
    'fy_end_date',
    'next_income_statement_report_date',
    'next_fy_end_date',
    'expected_report_date',
]
DAY_COUNT_COLS = [
    'feat_days_to_next_earnings',
    'feat_days_since_last_report',
    'feat_days_to_next_fy_end',
    'feat_days_to_next_report',
    'feat_days_to_expected_report',
    'feat_days_to_fy_end',
]

# Cast dates and drop rows missing the response or n_analysts
for c in DATE_COLS:
    price_target_df[c] = pd.to_datetime(price_target_df[c], errors='coerce')

model_df = price_target_df.loc[
    price_target_df['observed_target_pct'].notna()
    & price_target_df['n_analysts'].fillna(0).gt(0)
    ].copy().reset_index(drop=True)
print(f'Modelling rows (response present & n_analysts>0): {len(model_df)}')

# ---------------------------------------------------------------------------
# Categorical coords: every column tagged as `coord` in vw_pymc_feature_catalogue
# and present in pml.mv_pymc_price_target. Each becomes a model dimension with
# its own integer index array (`<coord>_idx`) and unique-label vector.
# ---------------------------------------------------------------------------
isin_labels = model_df['isin'].astype(str).values

CATEGORICAL_COORDS = [c for c in COORD_COLS
                      if c in model_df.columns and c not in ('isin', 'ticker')]
print(f'Categorical coords ({len(CATEGORICAL_COORDS)}): {CATEGORICAL_COORDS}')

coord_uniques = {}  # coord_name -> np.ndarray of unique labels
coord_idx = {}  # coord_name -> np.ndarray of int64 indices (len == n_obs)
for col in CATEGORICAL_COORDS:
    labels = model_df[col].fillna('Unknown').astype(str).values
    uniques, idx = np.unique(labels, return_inverse=True)
    coord_uniques[col] = uniques
    coord_idx[col] = idx.astype('int64')

# Back-compat aliases (legacy downstream cells expect `sector_*` symbols)
sector_uniques = coord_uniques['sector']
sector_idx = coord_idx['sector']
sector_labels = model_df['sector'].fillna('Unknown').astype(str).values

# ---------------------------------------------------------------------------
# 4.0c  Fiscal-anchor `time` coord for MvGaussianRandomWalk
# The six DATE columns give six ordered fiscal anchors per ISIN -> T=6
# random-walk steps. Build a per-isin day-offset matrix (N_isin, T) from
# t0 = income_statement_report_date and standardise it (Scaler demo).
# ---------------------------------------------------------------------------
FISCAL_ANCHORS = DATE_COLS  # ordered: report -> next earnings -> FY end -> ...
T = len(FISCAL_ANCHORS)

t0 = model_df[FISCAL_ANCHORS[0]]
t_days = np.stack(
    [(model_df[c] - t0).dt.days.to_numpy(dtype='float64') for c in FISCAL_ANCHORS],
    axis=1,
)  # shape (N_isin, T)

t_mean = np.nanmean(t_days)
t_std = np.nanstd(t_days) or 1.0
t_scaled = np.nan_to_num((t_days - t_mean) / t_std, nan=0.0)
print(f'Fiscal-anchor time tensor t_scaled: shape={t_scaled.shape} (N_isin, T)')

# ---------------------------------------------------------------------------
# 4.0d  Multivariate response Y (D-dimensional joint series)
# Broadcast cross-sectional snapshots across T anchors as a placeholder;
# swap with true lagged stacks (price_target_*_ago / target_pct_*_ago)
# once available for a fully longitudinal panel.
# ---------------------------------------------------------------------------
RESPONSE_COLS = ['feat_implied_upside', 'feat_target_dispersion_cv', 'observed_target_pct', 'observed_target_pct_med',
                 'price_target_high', 'price_target_low', 'price_target_median']
D = len(RESPONSE_COLS)

Y = np.stack(
    [np.tile(model_df[c].astype(float).to_numpy()[:, None], (1, T))
     for c in RESPONSE_COLS],
    axis=-1,
)  # shape (N_isin, T, D)

y_mean = Y.reshape(-1, D).mean(axis=0)
y_std = Y.reshape(-1, D).std(axis=0).clip(min=1e-6)
Y_std = (Y - y_mean) / y_std
print(f'Response tensor Y_std: shape={Y_std.shape} (N_isin, T, D); series={RESPONSE_COLS}')


In [ ]:
# 4.1  Predictor matrix — numeric feat_* with ≥70% coverage, capped at 8 features
num_pred_cols = [c for c in PREDICTOR_COLS if pd.api.types.is_numeric_dtype(model_df[c])]
# Force-include the new fiscal-calendar day-count predictors when present
for c in DAY_COUNT_COLS:
    if c in model_df.columns and c not in num_pred_cols:
        num_pred_cols.append(c)
coverage = model_df[num_pred_cols].notna().mean()
selected_predictors = coverage[coverage >= 0.70].sort_values(ascending=False).index.tolist()[:8]
print(f'Selected predictors ({len(selected_predictors)}): {selected_predictors}')

X_raw = model_df[selected_predictors].astype(float)
X_std = (X_raw - X_raw.mean()) / X_raw.std(ddof=0).replace(0, 1.0)
X = X_std.fillna(0.0).to_numpy()

# Response + precision-weight
y_target_pct = model_df['observed_target_pct'].astype(float).to_numpy()
n_analysts = model_df['n_analysts'].astype(int).clip(lower=1).to_numpy()


In [ ]:
# 4.2  PyMC model — Hierarchical Student-t + Multivariate Gaussian Random Walk
# Cross-sectional baseline (mu_isin) stays as the hierarchical Student-t mean;
# the MvGRW models shared temporal drift across the D-dim response
# (observed_target_pct, observed_target_pct_med, last_price) along the
# T fiscal anchors. See PyMC docs:
# https://www.pymc.io/projects/examples/en/latest/time_series/MvGaussianRandomWalk_demo.html
coords = {
    'isin': isin_labels,
    'time': np.arange(T),  # numeric index 0..T-1 (matches PyMC docs convention)
    'y_series': np.array(RESPONSE_COLS),
    'pt_feature': selected_predictors,
}
for _c in CATEGORICAL_COORDS:
    coords[_c] = coord_uniques[_c]

# Group-effect coords with enough rows per level to inform a hierarchical
# Normal sector/size/style intercept. Other categorical columns remain available
# as model coords / pm.Data for downstream slicing and posterior conditioning.
GROUP_EFFECTS = [c for c in ('industry', 'region', 'sector', 'size_class', 'style_class')
                 if c in CATEGORICAL_COORDS]

with pm.Model(coords=coords) as price_target_model:
    # ---------- pm.Data containers (names = MV aliases) ----------
    Y_obs = pm.Data('Y_obs', Y_std, dims=('isin', 'time', 'y_series'))
    t_obs = pm.Data('t_scaled', t_scaled, dims=('isin', 'time'))
    X_data = pm.Data('pt_features', X, dims=('isin', 'pt_feature'))
    # Precompute sqrt(n_analysts) in NumPy and pass as a float pm.Data
    # container. Doing `pt.sqrt(pt.cast(n_analysts_data, 'floatX'))` on an
    # integer pm.Data produces an inplace Sqrt op that nutpie / PyMC NUTS
    # reject with: 'Graph must not contain inplace operations: Sqrt(...)'.
    n_analysts_arr = np.asarray(n_analysts, dtype='float64')
    sqrt_n_analysts_arr = np.sqrt(np.maximum(n_analysts_arr, 1.0))
    n_analysts_data = pm.Data('n_analysts', n_analysts_arr, dims='isin')
    sqrt_n_analysts_data = pm.Data('sqrt_n_analysts', sqrt_n_analysts_arr, dims='isin')

    # One pm.Data per coord-indexed group (sector_idx, industry_idx, ...)
    idx_data_vars = {}
    for col in CATEGORICAL_COORDS:
        idx_data_vars[col] = pm.Data(f'{col}_idx', coord_idx[col], dims='isin')

    # ---------- Cross-sectional baseline (kept from the old model) ----------
    mu_global = pm.Normal('mu_global', mu=0.0, sigma=10.0)
    beta = pm.Normal('beta', mu=0.0, sigma=5.0, dims='pt_feature')

    group_effects = {}
    for col in GROUP_EFFECTS:
        sigma_g = pm.HalfNormal(f'sigma_{col}', sigma=10.0)
        z_g = pm.Normal(f'z_{col}', mu=0.0, sigma=1.0, dims=col)
        group_effects[col] = pm.Deterministic(
            f'{col}_effect', sigma_g * z_g, dims=col)

    eta = mu_global + pt.dot(X_data, beta)
    for col in GROUP_EFFECTS:
        eta = eta + group_effects[col][idx_data_vars[col]]
    mu_isin = pm.Deterministic('mu_isin', eta, dims='isin')

    # ---------- Gaussian Random Walks (intercept + slope), diagonal covariance ----------
    # NOTE: `pm.LKJCholeskyCov` (regardless of `sd_dist`) internally builds the
    # correlation matrix via the *onion method*, which is a Beta -> Normal chain.
    # PyTensor's stabilisation pass rewrites that `normal_rv(beta_rv.0, ...)`
    # into an inplace op, and both nutpie and the pure-Python PyMC NUTS sampler
    # then reject the compiled graph with:
    #   'Graph must not contain inplace operations:
    #    normal_rv{...}(beta_rv{...}.0, MakeVector{int64}.0, [0], [1])'.
    # Switching `sd_dist` to Exponential does NOT help — the offending op is
    # the LKJ correlation chain itself, not the scale prior.
    #
    # Workaround: drop the cross-series correlation and model each y_series'
    # random walk with an independent (diagonal) Gaussian innovation, in the
    # non-centred parameterisation (HalfNormal scale * standard Normal z).
    # This is the canonical NUTS-friendly parameterisation, mathematically
    # equivalent to an MvGaussianRandomWalk with a diagonal Cholesky factor,
    # and contains no Beta/Normal chain so PyTensor never produces an inplace
    # rewrite that NUTS rejects.
    sigma_alpha_innov = pm.HalfNormal(
        'sigma_alpha_innov', sigma=1.0, dims='y_series')
    sigma_beta_innov = pm.HalfNormal(
        'sigma_beta_innov', sigma=1.0, dims='y_series')
    z_alpha = pm.Normal(
        'z_alpha', mu=0.0, sigma=1.0, dims=('time', 'y_series'))
    z_beta = pm.Normal(
        'z_beta', mu=0.0, sigma=1.0, dims=('time', 'y_series'))
    alpha_innov = pm.Deterministic(
        'alpha_innov', z_alpha * sigma_alpha_innov[None, :],
        dims=('time', 'y_series'))
    beta_innov = pm.Deterministic(
        'beta_innov', z_beta * sigma_beta_innov[None, :],
        dims=('time', 'y_series'))
    alpha = pm.Deterministic(
        'alpha', pt.cumsum(alpha_innov, axis=0), dims=('time', 'y_series'))
    beta_t = pm.Deterministic(
        'beta_t', pt.cumsum(beta_innov, axis=0), dims=('time', 'y_series'))

    # Broadcast (T, D) walks over isin and add the cross-sectional baseline
    # (isin, time, y_series) = alpha[time] + beta_t[time] * t_obs[isin,time] + mu_isin
    regression = (
            alpha[None, :, :]
            + beta_t[None, :, :] * t_obs[:, :, None]
            + mu_isin[:, None, None]
    )

    # ---------- Heavy-tailed analyst noise, precision-weighted ----------
    # Use Exponential (not HalfNormal) for sigma_base. PyMC's HalfNormal is
    # internally `Abs(Normal)`, and PyTensor's stabilisation rewrites
    # `|x|` -> `sqrt(x**2)`. That `Sqrt(normal_rv)` then gets fused with the
    # `sqrt_n_analysts_data` divisor into a single Composite op that the
    # PyTensor inplace pass rewrites in place, after which NUTS rejects the
    # graph with: 'Graph must not contain inplace operations:
    # Composite{(i0 * (i1 / i2))}(Sqrt.0, normal_rv{...}.out, Sqrt.0)'.
    # Exponential(1.0) has the same support (R+) and similar prior mass on
    # O(1) scales for standardised Y, but its log-prob graph contains no
    # Normal/Sqrt, so the inplace fusion never triggers. This is the same
    # rationale as the LKJ `sd_dist=pm.Exponential.dist(...)` choice above.
    sigma_base = pm.Exponential('sigma_base', 1.0)  # Y is standardised
    # Use the precomputed sqrt(n_analysts) pm.Data container directly to
    # avoid the inplace Sqrt rewrite that breaks nutpie / PyMC NUTS.
    sigma_isin = pm.Deterministic(
        'sigma_isin',
        sigma_base / sqrt_n_analysts_data,
        dims='isin',
    )
    nu = pm.Gamma('nu', alpha=2.0, beta=0.1)  # ν > 0, mean 20

    pm.StudentT(
        'target_pct_obs',
        nu=nu,
        mu=regression,
        sigma=sigma_isin[:, None, None],
        observed=Y_obs,
        dims=('isin', 'time', 'y_series'),
    )

pm.model_to_graphviz(price_target_model)

## 5. Prior Predictive Checks

Sample from the prior to verify that priors imply a plausible distribution of analyst implied upside (`observed_target_pct`) centred near zero with heavy-tailed coverage of the empirical distribution.


In [ ]:
# 5.0  Prior predictive — covers the cross-sectional baseline (mu_isin),
# the diagonal GRW walks (alpha, beta_t) and their per-series innovation
# scales. (LKJ-Cholesky covariances were removed: their internal Beta→Normal
# graph triggers a PyTensor inplace rewrite that NUTS rejects — see §4.2.)
_prior_var_names = [
    "mu_global", "mu_isin", "beta",
    "alpha", "beta_t",
    "sigma_alpha_innov", "sigma_beta_innov",
    "sigma_base", "sigma_isin", "nu",
    "target_pct_obs",
]
with price_target_model:
    prior_idata = pm.sample_prior_predictive(
        draws=1000,
        var_names=_prior_var_names,
        random_seed=RANDOM_SEED,
        return_inferencedata=True,
    )
prior_idata


In [ ]:
# 5.1  Prior on μ_isin vs empirical observed_target_pct (cross-sectional baseline).
# Note: Y was standardised in §4.0d, so prior μ_isin lives on a z-scale.
empirical = price_target_df['observed_target_pct'].dropna().to_numpy()
empirical_z = (empirical - y_mean[0]) / y_std[0]  # match the Y_std scale
prior_mu = prior_idata.prior['mu_isin'].values.reshape(-1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(prior_mu, bins=60, density=True, alpha=0.6,
        color='#4FC3F7', edgecolor='#1e1e1e', label='prior μ_isin (z-scale)')
ax.hist(empirical_z, bins=60, density=True, alpha=0.5,
        color='#FFB74D', edgecolor='#1e1e1e', label='empirical (z-scale)')
ax.axvline(np.median(empirical_z), color='#FF5252', ls='--',
           label=f'empirical median z={np.median(empirical_z):.2f}')
ax.set_title('Prior predictive — μ_isin vs observed_target_pct (standardised)')
ax.set_xlabel('z-scored target_pct_avg')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 5.2  Prior predictive distribution of target_pct_obs.
# The likelihood now carries dims (isin, time, y_series); plot ECDFs per y_series
# so we can verify each response channel is plausibly covered by the prior.
from arviz_plots import plot_ppc_dist

for _series in RESPONSE_COLS:
    pc = plot_ppc_dist(
        prior_idata.sel(y_series=_series),
        group="prior_predictive",
        var_names=["target_pct_obs"],
        kind="ecdf",
        num_samples=500,
    )
    plt.suptitle(f"Prior predictive — {_series}")
    plt.tight_layout()
    plt.show()

In [ ]:
# 5.3  MvGRW prior sample paths (intercept walk `alpha` and slope walk `beta_t`).
# Each walk has shape (T, D); we overlay N prior draws per y_series so the
# joint temporal drift implied by the LKJ-Cholesky prior is visible.
# Reference: https://www.pymc.io/projects/examples/en/latest/time_series/MvGaussianRandomWalk_demo.html
_n_paths = 60
_time_coord = prior_idata.prior.coords['time'].values


# Flatten chain×draw and randomly thin to `_n_paths` paths.
def _stack_paths(da):
    arr = da.stack(sample=('chain', 'draw')).transpose('sample', 'time', 'y_series').values
    idx = rng.choice(arr.shape[0], size=min(_n_paths, arr.shape[0]), replace=False)
    return arr[idx]


alpha_paths = _stack_paths(prior_idata.prior['alpha'])  # (N, T, D)
beta_t_paths = _stack_paths(prior_idata.prior['beta_t'])  # (N, T, D)

fig, axes = plt.subplots(2, D, figsize=(4 * D, 6), sharex=True)
for d, series in enumerate(RESPONSE_COLS):
    # Row 0: intercept walk alpha[t, d]
    for path in alpha_paths:
        axes[0, d].plot(_time_coord, path[:, d], color='#4FC3F7', alpha=0.25, lw=0.8)
    axes[0, d].plot(_time_coord, alpha_paths.mean(0)[:, d],
                    color='#FF5252', lw=1.8, label='prior mean')
    axes[0, d].axhline(0, color='#888', ls=':', lw=0.8)
    axes[0, d].set_title(f'α (intercept walk) — {series}')
    axes[0, d].legend(fontsize=8)

    # Row 1: slope walk beta_t[t, d]
    for path in beta_t_paths:
        axes[1, d].plot(_time_coord, path[:, d], color='#66BB6A', alpha=0.25, lw=0.8)
    axes[1, d].plot(_time_coord, beta_t_paths.mean(0)[:, d],
                    color='#FF5252', lw=1.8, label='prior mean')
    axes[1, d].axhline(0, color='#888', ls=':', lw=0.8)
    axes[1, d].set_title(f'β_t (slope walk) — {series}')
    axes[1, d].set_xlabel('fiscal anchor (time)')
    axes[1, d].legend(fontsize=8)

axes[0, 0].set_ylabel('α prior draws')
axes[1, 0].set_ylabel('β_t prior draws')
plt.suptitle('Prior MvGRW sample paths — α & β_t across fiscal anchors', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 5.4  Prior per-series innovation scales for the diagonal GRW.
# (Replaces the previous LKJ-Cholesky correlation heatmap: cross-series
# correlations were dropped to keep the graph NUTS-friendly — see §4.2.)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, key, title in zip(
        axes,
        ('sigma_alpha_innov', 'sigma_beta_innov'),
        ('Prior σ — α walk innovations', 'Prior σ — β_t walk innovations'),
):
    arr = prior_idata.prior[key].stack(sample=('chain', 'draw')).values  # (y_series, sample)
    parts = ax.violinplot(arr.T, showmeans=True, showextrema=False)
    ax.set_xticks(range(1, D + 1))
    ax.set_xticklabels(RESPONSE_COLS, rotation=35, ha='right', fontsize=8)
    ax.set_title(title)
    ax.set_ylabel('σ')
plt.tight_layout()
plt.show()


## 6. Posterior Inference (NUTS)

Short NUTS run for notebook responsiveness — tune/draws can be increased for a production fit. The trace below feeds the posterior predictive checks and diagnostics in §7–§8.


In [ ]:
# Sampler dispatch — see §6 notes above for the nutpie / pymc fallback rationale.
# The MvGaussianRandomWalk + LKJCholeskyCov graph is fully supported by both
# the Rust nutpie sampler and PyMC's pure-Python NUTS; numpyro is skipped
# because MvGRW has no JAX dispatch.
sample_kwargs = dict(
    draws=100, tune=100, chains=4, cores=2,
    target_accept=0.95, random_seed=RANDOM_SEED,
    progressbar=True, return_inferencedata=True,
    idata_kwargs={"log_likelihood": False},
)

# Probe sampler availability up-front so we don't waste time on samplers
# whose backends aren't installed (e.g. nutpie's Rust binary missing
# yields a bare FileNotFoundError that is hard to diagnose downstream).
import importlib.util as _ilu

_candidate_samplers = []
if _ilu.find_spec("nutpie") is not None:
    _candidate_samplers.append("nutpie")
if _ilu.find_spec("numpyro") is not None:
    _candidate_samplers.append("numpyro")
_candidate_samplers.append("pymc")  # always available — pure-Python NUTS
print(f"Available NUTS samplers (in priority order): {_candidate_samplers}")

sampling_errors = []
idata = None
for _sampler in _candidate_samplers:
    try:
        with price_target_model:
            idata = pm.sample(nuts_sampler=_sampler, **sample_kwargs)
        print(f"Sampled successfully with nuts_sampler={_sampler!r}.")
        break
    except Exception as e:  # pragma: no cover - environment-dependent fallback
        sampling_errors.append((_sampler, repr(e)))
        print(f"nuts_sampler={_sampler!r} failed: {e!r}")

if idata is None:
    raise RuntimeError(
        "All NUTS samplers failed:\n"
        + "\n".join(f"  - {s}: {err}" for s, err in sampling_errors)
    )

# Merge prior / prior_predictive groups into the posterior idata.
from probabilistic_ml_model._pymc_arviz_compat import extend_datatree

idata = extend_datatree(idata, prior_idata)
idata


In [ ]:
# 6.1  Posterior MvGRW fan chart — α and β_t across fiscal anchors.
# Shows the 50% / 90% credible bands plus posterior median per y_series,
# matching the demo's "posterior trajectory" visualisation.
post = idata.posterior
_time_coord = post.coords['time'].values


def _bands(da):
    """Return (median, q05, q25, q75, q95) over chain×draw → (time, y_series)."""
    flat = da.stack(sample=('chain', 'draw'))
    return (
        flat.median('sample').values,
        flat.quantile(0.05, 'sample').values,
        flat.quantile(0.25, 'sample').values,
        flat.quantile(0.75, 'sample').values,
        flat.quantile(0.95, 'sample').values,
    )


alpha_stats = _bands(post['alpha'])
beta_t_stats = _bands(post['beta_t'])

fig, axes = plt.subplots(2, D, figsize=(4 * D, 6), sharex=True)
for d, series in enumerate(RESPONSE_COLS):
    for row, (med, q05, q25, q75, q95), name, colour in (
            (0, alpha_stats, 'α (intercept walk)', '#4FC3F7'),
            (1, beta_t_stats, 'β_t (slope walk)', '#66BB6A'),
    ):
        ax = axes[row, d]
        ax.fill_between(_time_coord, q05[:, d], q95[:, d], color=colour, alpha=0.20, label='90% CI')
        ax.fill_between(_time_coord, q25[:, d], q75[:, d], color=colour, alpha=0.40, label='50% CI')
        ax.plot(_time_coord, med[:, d], color='#FF5252', lw=1.8, label='median')
        ax.axhline(0, color='#888', ls=':', lw=0.8)
        ax.set_title(f'{name} — {series}')
        if row == 1:
            ax.set_xlabel('fiscal anchor (time)')
        ax.legend(fontsize=7, loc='best')

plt.suptitle('Posterior MvGRW trajectories — α & β_t (50% / 90% CI)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 6.2  Posterior per-series innovation scales for the diagonal GRW.
# (Replaces the previous LKJ-Cholesky correlation heatmap: cross-series
# correlations were dropped to keep the graph NUTS-friendly — see §4.2.)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, key, title in zip(
        axes,
        ('sigma_alpha_innov', 'sigma_beta_innov'),
        ('Posterior σ — α walk innovations', 'Posterior σ — β_t walk innovations'),
):
    arr = post[key].stack(sample=('chain', 'draw')).values  # (y_series, sample)
    parts = ax.violinplot(arr.T, showmeans=True, showextrema=False)
    ax.set_xticks(range(1, D + 1))
    ax.set_xticklabels(RESPONSE_COLS, rotation=35, ha='right', fontsize=8)
    ax.set_title(title)
    ax.set_ylabel('σ')
plt.tight_layout()
plt.show()


In [ ]:
# 6.3  Posterior predictive trajectories for a sampled ISIN.
# Pulls `target_pct_obs` from the posterior predictive and overlays observed
# Y_std against the credible bands per y_series along the fiscal anchors.
with price_target_model:
    pm.sample_posterior_predictive(
        idata, extend_inferencedata=True,
        random_seed=RANDOM_SEED, progressbar=True,
    )

_ppc = idata.posterior_predictive['target_pct_obs']  # (chain, draw, isin, time, y_series)
_sample_isin_idx = int(rng.integers(0, _ppc.sizes['isin']))
_sample_isin = str(_ppc.coords['isin'].values[_sample_isin_idx])

_slice = _ppc.isel(isin=_sample_isin_idx).stack(sample=('chain', 'draw'))
_med = _slice.median('sample').values  # (time, y_series)
_q05 = _slice.quantile(0.05, 'sample').values
_q95 = _slice.quantile(0.95, 'sample').values
_obs = Y_std[_sample_isin_idx]  # (time, y_series)

fig, axes = plt.subplots(1, D, figsize=(4.5 * D, 4), sharex=True)
for d, series in enumerate(RESPONSE_COLS):
    ax = axes[d]
    ax.fill_between(_time_coord, _q05[:, d], _q95[:, d],
                    color='#4FC3F7', alpha=0.30, label='90% PPC')
    ax.plot(_time_coord, _med[:, d], color='#FF5252', lw=1.6, label='PPC median')
    ax.plot(_time_coord, _obs[:, d], 'o-', color='#FFFFFF', lw=1.2, label='observed (z)')
    ax.set_title(f'{series}\nISIN={_sample_isin}')
    ax.set_xlabel('fiscal anchor (time)')
    ax.legend(fontsize=8)
axes[0].set_ylabel('standardised value')
plt.suptitle('Posterior predictive vs. observed (MvGRW reconstruction)', y=1.02)
plt.tight_layout()
plt.show()

## 7. Posterior Predictive Checks

Compare the predictive distribution of `observed_target_pct` (analyst implied upside) against the observed sample. A well-calibrated Student-t regression should cover the empirical density including its tails.


In [ ]:
with price_target_model:
    pm.sample_posterior_predictive(
        idata, extend_inferencedata=False,
        random_seed=RANDOM_SEED, progressbar=True,
    )

from arviz_plots import plot_ppc_dist

pc = plot_ppc_dist(
    idata,
    group="posterior_predictive",
    var_names=["target_pct_obs"],
    kind="ecdf",
    num_samples=200,
)
plt.suptitle("Posterior predictive — observed_target_pct")
plt.tight_layout()
plt.show()


In [ ]:
# 7.1 Posterior sector-mean μ vs empirical sector target_pct_avg
import pandas as pd

sector_agg = (
    model_df.groupby('sector', as_index=False)
    .agg(empirical_target_pct=('observed_target_pct', 'mean'),
         n_obs=('isin', 'size'))
    .sort_values('sector').reset_index(drop=True)
)

# Note: the model defines the hierarchical sector intercept as
# pm.Deterministic('sector_effect', sigma_sector * z_sector, dims='sector').
# There is no 'alpha_sector' variable in idata.posterior.
post_sector_effect = (
    idata.posterior['sector_effect']
    .mean(dim=('chain', 'draw'))
    .to_pandas()
    .rename('posterior_sector_effect')
    .reset_index()
)

by_sector = sector_agg.merge(post_sector_effect, on='sector', how='left')
print(by_sector)


## 8. MCMC Diagnostics

Convergence and sampler health: R-hat, ESS, divergences, trace plots, forest plot of predictor coefficients, and an ECDF of `p` — aligned with the ArviZ 1.0 diagnostic suite used in `probabilistic_ml_model.visualizations.arviz_diagnostics`.


In [ ]:
# 8.1  R-hat / ESS summary across the full upgraded parameter set.
# `requested` now covers:
#   - cross-sectional baseline:  mu_global, beta
#   - all 5 hierarchical effects (industry, region, sector, size_class, style_class)
#   - diagonal GRW components:   sigma_alpha_innov, sigma_beta_innov, alpha, beta_t
#   - likelihood scale / d.o.f.: sigma_base, nu
posterior = idata.posterior
requested = [
    'mu_global', 'beta', 'sigma_base', 'nu',
    'sigma_alpha_innov', 'sigma_beta_innov',
    'alpha', 'beta_t',
]
for _grp in GROUP_EFFECTS:
    requested.extend([f'sigma_{_grp}', f'{_grp}_effect'])

available, skipped = [], []
for v in requested:
    if v not in posterior.data_vars:
        skipped.append((v, 'not in posterior'))
        continue
    da = posterior[v]
    non_sample_sizes = [da.sizes[d] for d in da.dims if d not in ('chain', 'draw')]
    if any(s == 0 for s in non_sample_sizes):
        skipped.append((v, f'empty dim(s): {dict(da.sizes)}'))
        continue
    available.append(v)

if skipped:
    print('Skipping variables:')
    for name, reason in skipped:
        print(f'  - {name}: {reason}')

if not available:
    raise RuntimeError('No non-empty variables to summarise.')

summary = az.summary(idata, var_names=available, round_to=4)
summary.sort_values('r_hat', ascending=False).head(30)

In [ ]:
# 8.2  Divergences and aggregated R-hat / ESS.
# Per-variable max/min avoids the huge broadcast that to_array() would
# trigger over the new (time, y_series) GRW dims.
n_div = int(idata.sample_stats['diverging'].sum())


def _non_empty_vars(ds):
    keep = []
    for name, da in ds.data_vars.items():
        sizes = [da.sizes[d] for d in da.dims if d not in ('chain', 'draw')]
        if all(s > 0 for s in sizes):
            keep.append(name)
    return keep


posterior_tree = idata.posterior
posterior = posterior_tree.dataset if hasattr(posterior_tree, "dataset") else posterior_tree.to_dataset()

keep_vars = _non_empty_vars(posterior)
rhat_ds = az.rhat(posterior[keep_vars])
ess_ds = az.ess(posterior[keep_vars], method='bulk')

max_rhat = float(max(float(rhat_ds[v].max()) for v in rhat_ds.data_vars))
min_ess = float(min(float(ess_ds[v].min()) for v in ess_ds.data_vars))

# Surface the worst-offending variables for the new GRW block specifically,
# since cumsum(z * sigma) walks are the most prone to drift between chains.
_grw_keys = [v for v in ('alpha', 'beta_t', 'sigma_alpha_innov', 'sigma_beta_innov')
             if v in rhat_ds.data_vars]
_grw_report = {v: (float(rhat_ds[v].max()), float(ess_ds[v].min()))
               for v in _grw_keys}

print(f'Divergences: {n_div}')
print(f'Max R-hat:   {max_rhat:.4f}')
print(f'Min ESS:     {min_ess:.1f}')
if _grw_report:
    print('GRW block diagnostics (max R-hat, min ESS):')
    for v, (r, e) in _grw_report.items():
        print(f'  - {v:>20s}: r_hat={r:.3f}, ess_bulk={e:.1f}')

In [ ]:
# 8.3  Trace + density plots for the key parameters (manual, backend-agnostic).
# Expanded var set covers the diagonal-GRW innovation scales and all five
# hierarchical group-effect scales; the per-(time, y_series) `alpha` /
# `beta_t` arrays are intentionally summarised in §6.1's fan chart rather
# than blown up into 42 trace panels here.
import numpy as np
import matplotlib.pyplot as plt

_grw_scale_vars = ('sigma_alpha_innov', 'sigma_beta_innov')
_group_scale_vars = tuple(f'sigma_{g}' for g in GROUP_EFFECTS)

trace_vars = [v for v in (
    'mu_global', 'sigma_base', 'nu',
    *_grw_scale_vars,
    *_group_scale_vars,
    'beta',
) if v in available]


def _iter_param_slices(da):
    """Yield (label_suffix, 2D array of shape (chain, draw)) for each scalar
    slice of a posterior DataArray. Works for scalar params (no extra dims),
    vector params (one extra dim, e.g. 'sector', 'y_series'), and matrix
    params (multiple extra dims)."""
    extra_dims = [d for d in da.dims if d not in ('chain', 'draw')]
    if not extra_dims:
        yield '', da.values
        return
    stacked = da.stack(_extra=extra_dims)
    for i in range(stacked.sizes['_extra']):
        sl = stacked.isel(_extra=i)
        coord_bits = []
        for d in extra_dims:
            val = sl.coords[d].values.item() if d in sl.coords else i
            coord_bits.append(f"{d}={val}")
        yield '[' + ', '.join(coord_bits) + ']', sl.values  # (chain, draw)


if trace_vars:
    posterior = idata.posterior
    rows = []
    for v in trace_vars:
        if v not in posterior:
            continue
        da = posterior[v]
        if any(da.sizes[d] == 0 for d in da.dims):
            continue
        for suffix, arr in _iter_param_slices(da):
            rows.append((f"{v}{suffix}", arr))

    if not rows:
        print('No non-empty posterior variables to plot.')
    else:
        n = len(rows)
        fig, axes = plt.subplots(n, 2, figsize=(11, 2.2 * n), squeeze=False)
        for i, (label, arr) in enumerate(rows):
            ax_density, ax_trace = axes[i, 0], axes[i, 1]
            n_chains = arr.shape[0]
            for c in range(n_chains):
                vals = arr[c]
                vals = vals[np.isfinite(vals)]
                if vals.size == 0:
                    continue
                ax_density.hist(vals, bins=40, density=True,
                                histtype='step', linewidth=1.2,
                                label=f'chain {c}')
            ax_density.set_title(label)
            ax_density.set_ylabel('density')
            if n_chains > 1:
                ax_density.legend(fontsize=7, loc='best')
            for c in range(n_chains):
                ax_trace.plot(arr[c], linewidth=0.6, alpha=0.85,
                              label=f'chain {c}')
            ax_trace.set_title(label)
            ax_trace.set_xlabel('draw')
        plt.tight_layout()
        plt.show()
else:
    print('No trace-eligible variables available.')


In [ ]:
# 8.4 Energy plot (NUTS health) — single execution path, MvGRW-aware.
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# Robustly get groups for both InferenceData and DataTree
_groups_attr = getattr(idata, "groups", None)
if callable(_groups_attr):
    _groups = _groups_attr()  # classic arviz.InferenceData
else:
    _groups = tuple(_groups_attr) if _groups_attr is not None else ()  # xarray.DataTree

# DataTree group names start with '/', InferenceData uses plain names — normalize:
_group_names = {g.lstrip("/").split("/")[-1] for g in _groups}

if "sample_stats" in _group_names and "energy" in idata.sample_stats.data_vars:
    energy = idata.sample_stats["energy"]
    extra_dims = [d for d in energy.dims if d not in ("chain", "draw")]
    if extra_dims:
        energy = energy.mean(dim=extra_dims)  # collapse stray dims safely
    e = energy.values  # shape (chain, draw)

    marginal = (e - e.mean()).ravel()
    transition = np.diff(e, axis=1).ravel()
    transition = transition - transition.mean() + marginal.mean()

    # BFMI per chain (Betancourt 2016). Values < 0.3 typically indicate the
    # sampler is struggling — common with the cumsum-of-z GRW prior when
    # `sigma_*_innov` is poorly identified.
    bfmi = np.array([
        np.sum(np.diff(e[c]) ** 2) / np.sum((e[c] - e[c].mean()) ** 2)
        for c in range(e.shape[0])
    ])

    fig, ax = plt.subplots(figsize=(8, 4))
    grid = np.linspace(min(marginal.min(), transition.min()),
                       max(marginal.max(), transition.max()), 400)

    kde_m = gaussian_kde(marginal)
    kde_t = gaussian_kde(transition)
    ax.plot(grid, kde_m(grid), label="marginal energy", lw=2)
    ax.plot(grid, kde_t(grid), label="energy transition", lw=2)
    ax.fill_between(grid, kde_m(grid), alpha=0.25)
    ax.fill_between(grid, kde_t(grid), alpha=0.25)

    bfmi_str = ", ".join(f"chain {i}: {b:.2f}" for i, b in enumerate(bfmi))
    ax.set_title(f"Energy plot (BFMI — {bfmi_str})")
    ax.set_xlabel("energy")
    ax.set_ylabel("density")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("sample_stats.energy not available — skipping energy plot.")


In [ ]:
# 8.5  Forest plot of the diagonal-GRW innovation scales per y_series.
# A wider σ here means the corresponding response channel's random walk
# drifts more between fiscal anchors — useful for spotting which of the
# seven price-target / dispersion series carries the most temporal noise.
import arviz_plots as azp

for _key, _title in (
        ('sigma_alpha_innov', 'Intercept walk innovation σ (per y_series)'),
        ('sigma_beta_innov', 'Slope walk innovation σ (per y_series)'),
):
    if _key in idata.posterior.data_vars:
        azp.plot_forest(
            idata,
            var_names=[_key],
            combined=True,
        )
        plt.title(_title)
        plt.tight_layout()
        plt.show()
    else:
        print(f'{_key} not in posterior — skipped.')

### Summary

- §2 — EDA over `pml.mv_pymc_price_target` columns using `vw_pymc_feature_catalogue` (`pymc_role` / `feature_role` / `feature_alias`). Classification coords (`region, country, trading_country, exchange, unit, style_class, size_class, sector, industry`) profiled.
- §3 — per-column summary statistics merged with the catalogue, rolled up by `(pymc_role, feature_role, category)`; analyst-sentiment / valuation predictors visualised.
- §4 — hierarchical Student-t + **diagonal Gaussian Random Walk** PyMC model with `pm.Data` containers named after MV aliases (`Y_obs`, `t_scaled`, `pt_features`, `n_analysts`, `sqrt_n_analysts`, `<coord>_idx`), schema-aligned with `PriceTargetAchievement`. Likelihood dims are `(isin, time, y_series)` with D=7 response channels.
- §5 — prior predictive checks: μ_isin vs empirical (z-scale), per-`y_series` ECDFs, GRW sample paths for `alpha`/`beta_t`, and per-series innovation-scale violins.
- §6 — NUTS posterior inference with `nutpie → numpyro → pymc` fallback; posterior MvGRW fan charts, innovation-σ violins, and per-ISIN reconstruction overlays.
- §7 — per-`y_series` posterior predictive ECDFs, sector_effect vs empirical-z bars, and PPC-mean vs observed scatters (all D=7 channels).
- §8 — R-hat / ESS / divergences across the full expanded parameter set (`mu_global`, `beta`, `sigma_{industry,region,sector,size_class,style_class}`, `sigma_{alpha,beta}_innov`, `alpha`, `beta_t`, `sigma_base`, `nu`); trace + density panels, energy plot with BFMI, and a forest plot of the GRW innovation scales.